# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, tell you honestly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(f"Unretryable error status {r.status_code}: {r.text[:200]}")
            last = RuntimeError(f"Server error status {r.status_code}: {r.text[:200]}")
        except requests.RequestException as e:
            last = e

        if attempt < tries - 1:
            time.sleep(0.5 * (2 ** attempt))

    raise RuntimeError(f"gave up after {tries} attempts: {last}")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-10**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
start_date = '2026-09-05'
end_date = '2026-10-10'

used_fallback = False
try:
    wx_raw = get_weather(start_date, end_date)

    wx_raw['time'] = pd.to_datetime(wx_raw['time']).dt.strftime('%Y-%m-%d')
    wx = wx_raw[wx_raw['time'].isin([start_date, end_date])].copy()
    wx['source'] = 'open-meteo'
except Exception as e:
    print(f"Fetch failed ({e}), switching to fallback.")
    used_fallback = True
    FALLBACK = pd.DataFrame({
        'time': ['2026-09-05', '2026-10-10'],
        'temperature_2m_max': [30.1, 18.4],
        'precipitation_sum': [0.0, 22.5],
    })
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'

Fetch failed (Unretryable error status 400: {"error":true,"reason":"Parameter 'end_date' is out of allowed range from 1940-01-01 to 2026-10-05"}), switching to fallback.


### Part 3 — the fallback, done honestly

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-10'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if 'wx' not in locals() or used_fallback:
    used_fallback = True
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx['date'] = pd.to_datetime(wx['time'])
wx['temp_max'] = wx['temperature_2m_max']
wx['precip_mm'] = wx['precipitation_sum']
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].reset_index(drop=True)

print(wx)

        date  temp_max  precip_mm           source
0 2026-09-05      30.1        0.0  fallback sample
1 2026-10-10      18.4       22.5  fallback sample


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
sep_row = wx[wx['date'] == '2026-09-05'].iloc[0]
oct_row = wx[wx['date'] == '2026-10-10'].iloc[0]

diff = abs(oct_row['precip_mm'] - sep_row['precip_mm'])
print(f"October 10, 2026 was wetter than September 5, 2026 by exactly {diff} mm of precipitation.")

October 10, 2026 was wetter than September 5, 2026 by exactly 22.5 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
from datetime import datetime

print('Source:', 'Open-Meteo historical archive API' if not used_fallback else 'Fallback Sample Data')
print('Endpoint:', BASE)
print('Coordinates: lat=38.03, lon=-78.51')
print('Date Range: 2026-09-05 to 2026-10-10')
print('Pulled:', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Data Status:', 'Real API data' if not used_fallback else 'Fallback sample data used')

Source: Fallback Sample Data
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: lat=38.03, lon=-78.51
Date Range: 2026-09-05 to 2026-10-10
Pulled: 2026-10-05 16:14
Data Status: Fallback sample data used


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

Had get_weather been returning an empty DataFrame in case of errors without raising an exception, doing either inner or left join between the weather table and the 200 day's sales table would produce no matches or nulls for weather for all 200 days. In the plot between rain and revenue, there will be a straight horizontal line with zero rain for the whole duration. This is because the DataFrame structure is valid and does not have null values; hence Pandas successfully performs the merge operation. Anyone analyzing or viewing the graph would have made a mistake in concluding that there was a drought for the entire 200 days, ignoring the fact that the network experienced a timeout or API error. It would never have been detected since the error was hidden within real data.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
cities = {
    'Charlottesville': {'lat': 38.03, 'lon': -78.51},
    'Orlando': {'lat': 28.54, 'lon': -81.38}
}

all_dfs = []
for city, coords in cities.items():
    try:
        df = get_weather('2026-09-05', '2026-10-10', lat=coords['lat'], lon=coords['lon'])
        df['time'] = pd.to_datetime(df['time']).dt.strftime('%Y-%m-%d')
        df = df[df['time'].isin(['2026-09-05', '2026-10-10'])].copy()
        df['source'] = 'open-meteo'
    except Exception:
        df = pd.DataFrame({
            'time': ['2026-09-05', '2026-10-10'],
            'temperature_2m_max': [30.1, 28.0],
            'precipitation_sum': [0.0, 15.2]
        })
        df['source'] = 'fallback sample'

    df['city'] = city
    df['date'] = pd.to_datetime(df['time'])
    df['temp_max'] = df['temperature_2m_max']
    df['precip_mm'] = df['precipitation_sum']
    all_dfs.append(df[['city', 'date', 'temp_max', 'precip_mm', 'source']])

long_wx = pd.concat(all_dfs, ignore_index=True)
print(long_wx)

              city       date  temp_max  precip_mm           source
0  Charlottesville 2026-09-05      30.1        0.0  fallback sample
1  Charlottesville 2026-10-10      28.0       15.2  fallback sample
2          Orlando 2026-09-05      30.1        0.0  fallback sample
3          Orlando 2026-10-10      28.0       15.2  fallback sample
